# 03 Offline Augmentation

Generate realistic offline augmentation from the training split only.


## Purpose

Notebook 03 creates train-only augmented data for factory sign detection: geometric, photometric, weather/quality, and a safe synthetic-placement placeholder.

Validation and test splits must remain original and untouched. This notebook does not train models, build ablation datasets, or modify any original split files.


## 1. Setup

Find the `sign-detection` root, add it to `sys.path`, and import train-only augmentation helpers.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display
from PIL import Image, ImageDraw


# Notebook execution can start from the repository root or from notebooks/.
# This helper finds the sign-detection root before importing project modules.
def find_project_root(start: Path) -> Path:
    """Return the sign-detection project root from a notebook working directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "augmentation_config.yaml").exists() and (candidate / "src").exists():
            return candidate
        sign_child = candidate / "sign-detection"
        if (sign_child / "configs" / "augmentation_config.yaml").exists() and (sign_child / "src").exists():
            return sign_child
    raise RuntimeError("Could not find sign-detection project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Add project root so reusable src/ augmentation modules import cleanly in Jupyter.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep this notebook as orchestration only. The augmentation logic lives in src/augmentation/.
from src.augmentation.offline_geometric import generate_geometric_augmentation
from src.augmentation.offline_photometric import generate_photometric_augmentation
from src.augmentation.offline_weather_quality import generate_weather_quality_augmentation
from src.augmentation.synthetic_placement import generate_synthetic_placement_augmentation
from src.augmentation.offline_common import find_image_label_pairs

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Github\smart-factory-safety-monitoring-system\sign-detection


## 2. Load Configuration

Ratios and output paths come from config files. Paths are resolved relative to `sign-detection`.


In [2]:
# Load dataset paths and augmentation ratios from config files.
# This keeps the notebook portable and prevents hardcoded absolute paths.
with (PROJECT_ROOT / "configs" / "dataset_config.yaml").open("r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)
with (PROJECT_ROOT / "configs" / "augmentation_config.yaml").open("r", encoding="utf-8") as file:
    augmentation_config = yaml.safe_load(file)

paths = dataset_config["paths"]
offline_config = augmentation_config["offline_augmentation"]
seed = int(dataset_config.get("random_seed", 42))

# Notebook 03 intentionally reads only the train split generated by Notebook 02.
# Validation and test paths are never loaded here, so they cannot be augmented by mistake.
splits_dir = PROJECT_ROOT / paths["splits_original"]
train_images_dir = splits_dir / "train" / "images"
train_labels_dir = splits_dir / "train" / "labels"

# Augmented samples are generated artifacts, separate from original train images/labels.
augmented_train_dir = PROJECT_ROOT / paths["augmented_train"]
reports_dir = PROJECT_ROOT / paths.get("reports_augmentation", "reports/augmentation")
figures_dir = PROJECT_ROOT / paths.get("reports_figures", "reports/figures")

# Reports and figures are safe to create; they do not alter source data.
reports_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("Train images:", train_images_dir)
print("Train labels:", train_labels_dir)
print("Augmented output root:", augmented_train_dir)
print("Random seed:", seed)


Train images: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\splits_original\train\images
Train labels: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\splits_original\train\labels
Augmented output root: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train
Random seed: 42


## 3. Verify Train Split and Preview Counts

Only `data/generated/splits_original/train/` is read. Validation and test folders are intentionally not referenced.


In [3]:
# Fail early if Notebook 02 has not produced the train split yet.
if not train_images_dir.exists() or not train_labels_dir.exists():
    raise FileNotFoundError(
        "Missing train split folders. Run Notebook 02 before Notebook 03."
    )

# Pair images with same-stem YOLO label files. Empty labels are valid no-sign samples.
train_pairs = find_image_label_pairs(train_images_dir, train_labels_dir)
if not train_pairs:
    raise ValueError(
        "No train image-label pairs found. Run Notebook 02 and review split outputs."
    )


def estimated_count(total: int, ratio: float) -> int:
    """Estimate selected image count using the same ratio policy as augmentation helpers."""
    if ratio <= 0 or total == 0:
        return 0
    return max(1, min(total, int(round(total * ratio))))


# Preview expected output counts before writing any augmented files.
ratio_preview = pd.DataFrame(
    [
        {
            "augmentation_type": "geometric",
            "ratio": offline_config.get("geometric_ratio", 0.0),
        },
        {
            "augmentation_type": "photometric",
            "ratio": offline_config.get("photometric_ratio", 0.0),
        },
        {
            "augmentation_type": "weather_quality",
            "ratio": offline_config.get("weather_quality_ratio", 0.0),
        },
        {
            "augmentation_type": "synthetic_placement",
            "ratio": offline_config.get("synthetic_placement_ratio", 0.0),
        },
    ]
)
ratio_preview["estimated_selected_images"] = ratio_preview["ratio"].apply(lambda value: estimated_count(len(train_pairs), float(value)))

print(f"Original train image-label pairs: {len(train_pairs)}")
display(ratio_preview)


Original train image-label pairs: 332


,augmentation_type,ratio,estimated_selected_images
0,geometric,0.30,100
1,photometric,0.25,83
2,weather_quality,0.20,66
3,synthetic_placement,0.00,0


## 4. Configure Output Folders and Overwrite Safety

Augmented files are written under `data/generated/augmented_train/`. Keep `OVERWRITE_AUGMENTED = False` for normal use.


In [4]:
# Safety switch: keep False for normal runs.
# If outputs already exist and this is False, augmentation helpers skip those files and report them.
OVERWRITE_AUGMENTED = False

# Each augmentation family writes to its own image/label folder so Notebook 04 can combine them cleanly.
output_paths = {
    "geometric": {
        "images": augmented_train_dir / "geometric" / "images",
        "labels": augmented_train_dir / "geometric" / "labels",
    },
    "photometric": {
        "images": augmented_train_dir / "photometric" / "images",
        "labels": augmented_train_dir / "photometric" / "labels",
    },
    "weather_quality": {
        "images": augmented_train_dir / "weather_quality" / "images",
        "labels": augmented_train_dir / "weather_quality" / "labels",
    },
    "synthetic_placement": {
        "images": augmented_train_dir / "synthetic_placement" / "images",
        "labels": augmented_train_dir / "synthetic_placement" / "labels",
    },
}

print("Overwrite augmented outputs:", OVERWRITE_AUGMENTED)
for augmentation_type, folders in output_paths.items():
    print(augmentation_type, "->", folders["images"], "and", folders["labels"])


Overwrite augmented outputs: False
geometric -> C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train\geometric\images and C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train\geometric\labels
photometric -> C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train\photometric\images and C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train\photometric\labels
weather_quality -> C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train\weather_quality\images and C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train\weather_quality\labels
synthetic_placement -> C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train\synthetic_placement\images and C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\

## 5. Generate Geometric Augmentation

Geometric transforms move signs, so labels are updated by transforming bbox corners and clipping boxes to the image canvas.


In [5]:
# Geometric augmentation changes object positions, so its helper transforms YOLO bbox corners.
# If all boxes become invalid after transform/clipping, the sample is skipped and recorded in the report.
geometric_report_df = generate_geometric_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=output_paths["geometric"]["images"],
    output_labels_dir=output_paths["geometric"]["labels"],
    ratio=float(offline_config.get("geometric_ratio", 0.0)),
    config=augmentation_config.get("geometric", {}),
    seed=seed,
    overwrite=OVERWRITE_AUGMENTED,
)

# Save one row per selected image so skipped/failed samples are easy to audit.
geometric_report_path = reports_dir / "geometric_augmentation_report.csv"
geometric_report_df.to_csv(geometric_report_path, index=False)
print(f"Saved geometric report: {geometric_report_path}")
display(geometric_report_df.head())


Saved geometric report: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\augmentation\geometric_augmentation_report.csv


,augmentation_type,original_image_path,original_label_path,augmented_image_path,augmented_label_path,status,notes,num_original_objects,num_augmented_objects
0,geometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,affine rotation/scale/translation; perspective...,1,1
1,geometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,affine rotation/scale/translation; perspective...,1,1
2,geometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,affine rotation/scale/translation; perspective...,1,1
3,geometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,affine rotation/scale/translation; perspective...,1,1
4,geometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,affine rotation/scale/translation; perspective...,1,1


## 6. Generate Photometric Augmentation

Photometric transforms alter lighting and color only. Object geometry is unchanged, so labels are copied unchanged, including empty no-sign labels.


In [6]:
# Photometric augmentation changes lighting/color only, so labels are copied unchanged.
# This includes empty no-sign labels, which remain empty after augmentation.
photometric_report_df = generate_photometric_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=output_paths["photometric"]["images"],
    output_labels_dir=output_paths["photometric"]["labels"],
    ratio=float(offline_config.get("photometric_ratio", 0.0)),
    config=augmentation_config.get("photometric", {}),
    seed=seed,
    overwrite=OVERWRITE_AUGMENTED,
)

photometric_report_path = reports_dir / "photometric_augmentation_report.csv"
photometric_report_df.to_csv(photometric_report_path, index=False)
print(f"Saved photometric report: {photometric_report_path}")
display(photometric_report_df.head())


Saved photometric report: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\augmentation\photometric_augmentation_report.csv


,augmentation_type,original_image_path,original_label_path,augmented_image_path,augmented_label_path,status,notes,num_original_objects,num_augmented_objects
0,photometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=low_light,1,1
1,photometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=low_light,1,1
2,photometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=ir_grayscale,1,1
3,photometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=ir_grayscale,1,1
4,photometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=ir_grayscale,1,1


## 7. Generate Weather / Quality Augmentation

Weather and quality transforms simulate CCTV artifacts such as blur, compression, noise, dirty lens spots, and low resolution. Labels remain unchanged.


In [7]:
# Weather/quality augmentation simulates CCTV degradation without moving objects.
# Labels are therefore copied unchanged, just like photometric augmentation.
weather_quality_report_df = generate_weather_quality_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=output_paths["weather_quality"]["images"],
    output_labels_dir=output_paths["weather_quality"]["labels"],
    ratio=float(offline_config.get("weather_quality_ratio", 0.0)),
    config=augmentation_config.get("weather_quality", {}),
    seed=seed,
    overwrite=OVERWRITE_AUGMENTED,
)

weather_quality_report_path = reports_dir / "weather_quality_augmentation_report.csv"
weather_quality_report_df.to_csv(weather_quality_report_path, index=False)
print(f"Saved weather/quality report: {weather_quality_report_path}")
display(weather_quality_report_df.head())


Saved weather/quality report: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\augmentation\weather_quality_augmentation_report.csv


,augmentation_type,original_image_path,original_label_path,augmented_image_path,augmented_label_path,status,notes,num_original_objects,num_augmented_objects
0,weather_quality,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=gaussian_blur,1,1
1,weather_quality,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=sensor_noise...,1,1
2,weather_quality,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=gaussian_blu...,1,1
3,weather_quality,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=low_resoluti...,1,1
4,weather_quality,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,label copied unchanged; transform=gaussian_blu...,1,1


## 8. Synthetic Placement Placeholder

Synthetic placement is disabled by default. It is kept as a placeholder because future copy-paste augmentation must update labels correctly.


In [8]:
# Synthetic placement is intentionally a placeholder for now.
# Copy-paste style augmentation would change labels and must be implemented carefully later.
synthetic_report_df = generate_synthetic_placement_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=output_paths["synthetic_placement"]["images"],
    output_labels_dir=output_paths["synthetic_placement"]["labels"],
    ratio=float(offline_config.get("synthetic_placement_ratio", 0.0)),
    config=augmentation_config.get("synthetic_placement", {}),
    seed=seed,
    overwrite=OVERWRITE_AUGMENTED,
)

synthetic_report_path = reports_dir / "synthetic_placement_report.csv"
synthetic_report_df.to_csv(synthetic_report_path, index=False)
print(f"Saved synthetic placement report: {synthetic_report_path}")
display(synthetic_report_df)


Saved synthetic placement report: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\augmentation\synthetic_placement_report.csv


,augmentation_type,original_image_path,original_label_path,augmented_image_path,augmented_label_path,status,notes,num_original_objects,num_augmented_objects
0,synthetic_placement,,,,,skipped,synthetic placement disabled because ratio is 0.0,0,0


## 9. Build Offline Augmentation Summary

The summary shows how many selected samples were generated, skipped, or failed for each augmentation group.


In [9]:
def summarize_report(report_df: pd.DataFrame, augmentation_type: str) -> dict:
    """Summarize one augmentation report for the final CSV."""
    # Empty reports can happen when a ratio is 0.0 or no train pairs exist.
    if report_df.empty:
        return {
            "augmentation_type": augmentation_type,
            "selected_images": 0,
            "generated_images": 0,
            "skipped_images": 0,
            "failed_images": 0,
            "original_objects": 0,
            "augmented_objects": 0,
            "notes": "no images selected",
        }

    # Status counts make reruns easy to understand: generated vs skipped existing outputs vs failures.
    statuses = report_df["status"].value_counts()
    notes = "; ".join(sorted(set(report_df["notes"].dropna().astype(str))))
    return {
        "augmentation_type": augmentation_type,
        "selected_images": int(len(report_df)),
        "generated_images": int(statuses.get("generated", 0)),
        "skipped_images": int(statuses.get("skipped", 0)),
        "failed_images": int(statuses.get("failed", 0)),
        "original_objects": int(report_df["num_original_objects"].fillna(0).sum()),
        "augmented_objects": int(report_df["num_augmented_objects"].fillna(0).sum()),
        "notes": notes,
    }


# Combine all augmentation-family reports into one compact summary for Notebook 04 readiness.
summary_df = pd.DataFrame(
    [
        summarize_report(geometric_report_df, "geometric"),
        summarize_report(photometric_report_df, "photometric"),
        summarize_report(weather_quality_report_df, "weather_quality"),
        summarize_report(synthetic_report_df, "synthetic_placement"),
    ]
)

summary_path = reports_dir / "offline_augmentation_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved offline augmentation summary: {summary_path}")
display(summary_df)


Saved offline augmentation summary: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\augmentation\offline_augmentation_summary.csv


,augmentation_type,selected_images,generated_images,skipped_images,failed_images,original_objects,augmented_objects,notes
0,geometric,100,100,0,0,103,103,affine rotation/scale/translation; perspective...
1,photometric,83,83,0,0,81,81,label copied unchanged; transform=ir_grayscale...
2,weather_quality,66,66,0,0,62,62,label copied unchanged; transform=dirty_lens; ...
3,synthetic_placement,1,0,1,0,0,0,synthetic placement disabled because ratio is 0.0


## 10. Optional Visual Verification Grid

This grid samples generated images from each augmentation type for a quick visual sanity check.


In [10]:
def save_visual_grid(
    report_frames: list[pd.DataFrame], output_path: Path, max_per_type: int = 2
) -> Path:
    """Save one compact image grid from generated augmentation samples."""
    # Pull a small number of generated examples from each augmentation family.
    rows = []
    for report_df in report_frames:
        if report_df.empty:
            continue
        generated = report_df[report_df["status"] == "generated"].head(max_per_type)
        rows.extend(generated.to_dict("records"))
    if not rows:
        return output_path

    # Build simple fixed-size tiles so the output is readable in reports.
    tiles = []
    for row in rows:
        image_path = Path(row["augmented_image_path"])
        if not image_path.exists():
            continue
        with Image.open(image_path) as image:
            thumb = image.convert("RGB")
            thumb.thumbnail((260, 180))
            tile = Image.new("RGB", (280, 220), "white")
            tile.paste(thumb, ((280 - thumb.width) // 2, 10))
            draw = ImageDraw.Draw(tile)
            draw.text((10, 192), str(row["augmentation_type"]), fill="#222222")
            draw.text((10, 206), image_path.name[:38], fill="#555555")
            tiles.append(tile)

    if not tiles:
        return output_path

    # Arrange tiles in up to three columns to keep the figure compact.
    cols = min(3, len(tiles))
    grid_rows = (len(tiles) + cols - 1) // cols
    grid = Image.new("RGB", (cols * 280, grid_rows * 220), "white")
    for index, tile in enumerate(tiles):
        grid.paste(tile, ((index % cols) * 280, (index // cols) * 220))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    grid.save(output_path)
    return output_path


visual_grid_path = save_visual_grid(
    [geometric_report_df, photometric_report_df, weather_quality_report_df],
    figures_dir / "offline_augmentation_examples.png",
)
if visual_grid_path.exists():
    print(f"Saved visual grid: {visual_grid_path}")
else:
    print("Skipped visual grid because no generated images were available.")


Saved visual grid: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\figures\offline_augmentation_examples.png


## 11. Notebook 04 Checklist

Before moving to Notebook 04:

- Confirm augmentation reports were saved under `reports/augmentation/`.
- Review skipped or failed rows before using augmented data.
- Inspect `reports/figures/offline_augmentation_examples.png` if generated.
- Confirm validation and test split folders were not touched.
- Use Notebook 04 to build ablation datasets from original and augmented training data.
